# Prepare 20K Demo Dataset

This notebook creates a **20,000-sample balanced demo dataset** (10K benign + 10K attack) from `checkpoint_balanced_full.parquet`.

Outputs:
- `X_test_demo_20k.csv` — normalized feature matrix (20K rows)
- `y_test_demo_20k.csv` — labels (0 = benign, 1 = attack)

Files are saved to **both** Google Drive and the local Colab kernel path.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import json
import numpy as np
import pandas as pd

# -------------------------------------------------------
# 1. LOCATE THE PARQUET FILE
# -------------------------------------------------------
parquet_candidates = [
    '/content/drive/MyDrive/IDS_Dashboard_Submission/models/checkpoint_balanced_full.parquet',
    '/content/drive/MyDrive/test/checkpoint_balanced_full.parquet',
    '/content/drive/MyDrive/checkpoint_balanced_full.parquet',
    'c:/Users/Trumpler/Downloads/models-20260613T064206Z-3-001/models/checkpoint_balanced_full.parquet',
]

PARQUET_FILE = None
for p in parquet_candidates:
    if os.path.exists(p):
        PARQUET_FILE = p
        break

if PARQUET_FILE is None:
    raise FileNotFoundError(
        'checkpoint_balanced_full.parquet not found. '
        'Make sure Google Drive is mounted and the file is in one of the candidate paths above.'
    )
print(f'Found parquet at: {PARQUET_FILE}')

Found parquet at: /content/drive/MyDrive/IDS_Dashboard_Submission/models/checkpoint_balanced_full.parquet


In [3]:
# -------------------------------------------------------
# 2. LOCATE THE BOUNDS FILE (for normalisation)
# -------------------------------------------------------
bounds_candidates = [
    '/content/drive/MyDrive/IDS_Dashboard_Submission/models/X_bounds_cic.json',
    '/content/drive/MyDrive/test/X_bounds_cic.json',
    '/content/drive/MyDrive/X_bounds_cic.json',
    'c:/Users/Trumpler/Downloads/models-20260613T064206Z-3-001/models/X_bounds_cic.json',
]

BOUNDS_FILE = None
for p in bounds_candidates:
    if os.path.exists(p):
        BOUNDS_FILE = p
        break

if BOUNDS_FILE is None:
    raise FileNotFoundError(
        'X_bounds_cic.json not found. '
        'This file is required to reconstruct the same normalisation as the trained model.'
    )
print(f'Found bounds at: {BOUNDS_FILE}')

Found bounds at: /content/drive/MyDrive/IDS_Dashboard_Submission/models/X_bounds_cic.json


In [4]:
# -------------------------------------------------------
# 3. LOAD PARQUET AND CAST TYPES
# -------------------------------------------------------
print('Loading parquet...')
df = pd.read_parquet(PARQUET_FILE)
print(f'Total rows: {len(df):,}')

for col in df.columns:
    if col == 'Label':
        df[col] = df[col].astype(np.int32)
    else:
        df[col] = df[col].astype(np.float32)

feature_cols = df.columns.drop('Label').tolist()
print(f'Features: {len(feature_cols)} columns')
print(f'Benign rows: {(df["Label"]==0).sum():,}')
print(f'Attack rows: {(df["Label"]==1).sum():,}')

Loading parquet...
Total rows: 5,493,868
Features: 77 columns
Benign rows: 2,746,934
Attack rows: 2,746,934


In [5]:
# -------------------------------------------------------
# 4. SAMPLE 10K BENIGN + 10K ATTACK (same split logic as original)
# -------------------------------------------------------
SAMPLE_PER_CLASS = 10_000

benign_idx = df[df['Label'] == 0].index.values.copy()
attack_idx = df[df['Label'] == 1].index.values.copy()

np.random.seed(42)
np.random.shuffle(benign_idx)
np.random.shuffle(attack_idx)

# Use the same partition offsets as prepare_demo_data.py (skip training split)
# Train split consumed indices [0:1576063], test slice starts at [1576063:]
test_benign_idx = benign_idx[1576063 : 1576063 + 1170871]
test_attack_idx = attack_idx[1576063 : 1576063 + 143849]

# Sample from test partition
test_benign_sample = df.loc[test_benign_idx].head(SAMPLE_PER_CLASS)
test_attack_sample = df.loc[test_attack_idx].head(SAMPLE_PER_CLASS)

print(f'Benign sample size: {len(test_benign_sample):,}')
print(f'Attack sample size: {len(test_attack_sample):,}')

demo_df = pd.concat([test_benign_sample, test_attack_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Combined demo size: {len(demo_df):,} rows')

Benign sample size: 10,000
Attack sample size: 10,000
Combined demo size: 20,000 rows


In [6]:
# -------------------------------------------------------
# 5. RECONSTRUCT NORMALISATION SCALER FROM BOUNDS FILE
#    (same method as prepare_demo_data.py)
# -------------------------------------------------------
print(f'Loading bounds from: {BOUNDS_FILE}')
with open(BOUNDS_FILE, 'r') as f:
    bounds = json.load(f)

means, stds = [], []
for col in feature_cols:
    raw_min = float(df[col].min())
    raw_max = float(df[col].max())
    norm_min = bounds[col]['min']
    norm_max = bounds[col]['max']
    if abs(norm_max - norm_min) < 1e-6:
        std = 1e-6
        mean = raw_min
    else:
        std = (raw_max - raw_min) / (norm_max - norm_min)
        mean = raw_max - norm_max * std
        if std < 0:
            std = abs(std)
            mean = raw_max - norm_max * std
    means.append(mean)
    stds.append(std)

means = np.array(means, dtype=np.float32)
stds  = np.array(stds,  dtype=np.float32)
print('Scaler reconstructed from bounds file.')

Loading bounds from: /content/drive/MyDrive/IDS_Dashboard_Submission/models/X_bounds_cic.json
Scaler reconstructed from bounds file.


In [7]:
# -------------------------------------------------------
# 6. NORMALISE
# -------------------------------------------------------
X_demo_raw = demo_df[feature_cols].values
y_demo     = demo_df['Label'].values

X_demo_norm = (X_demo_raw - means) / stds

X_demo_df = pd.DataFrame(X_demo_norm, columns=feature_cols)
y_demo_df = pd.DataFrame(y_demo, columns=['Label'])

print(f'X shape: {X_demo_df.shape}')
print(f'y shape: {y_demo_df.shape}')
print(f'Benign count: {(y_demo==0).sum():,}')
print(f'Attack count: {(y_demo==1).sum():,}')

X shape: (20000, 77)
y shape: (20000, 1)
Benign count: 10,000
Attack count: 10,000


In [ ]:
# -------------------------------------------------------
# 7. SAVE TO GOOGLE DRIVE (primary) AND KERNEL (backup)
# -------------------------------------------------------

# --- Google Drive output path ---
# Mirrors the existing demo folder structure on Drive
gdrive_candidates = [
    '/content/drive/MyDrive/IDS_Dashboard_Submission/datasets/demo',
    '/content/drive/MyDrive/IDS_Dashboard_Submission/datasets',
    '/content/drive/MyDrive/test',
    '/content/drive/MyDrive',
]
GDRIVE_OUT = None
for p in gdrive_candidates:
    if os.path.exists(p):
        GDRIVE_OUT = p
        break

if GDRIVE_OUT is None:
    GDRIVE_OUT = '/content/drive/MyDrive'
    os.makedirs(GDRIVE_OUT, exist_ok=True)

gdrive_x = os.path.join(GDRIVE_OUT, 'X_test_demo_20k.csv')
gdrive_y = os.path.join(GDRIVE_OUT, 'y_test_demo_20k.csv')

X_demo_df.to_csv(gdrive_x, index=False)
y_demo_df.to_csv(gdrive_y, index=False)
print(f'Saved to Google Drive:')
print(f'  X -> {gdrive_x}')
print(f'  y -> {gdrive_y}')

# --- Kernel backup path ---
kernel_out = '/content/demo'
os.makedirs(kernel_out, exist_ok=True)
X_demo_df.to_csv(os.path.join(kernel_out, 'X_test_demo_20k.csv'), index=False)
y_demo_df.to_csv(os.path.join(kernel_out, 'y_test_demo_20k.csv'), index=False)
print(f'\nAlso saved to kernel at: {kernel_out}')

print(f'\nDone! {len(X_demo_df):,} samples saved (10K benign + 10K attack).')

Saved to Google Drive:
  X -> /content/drive/MyDrive/IDS_Dashboard_Submission/datasets/demo/X_test_demo_20k.csv
  y -> /content/drive/MyDrive/IDS_Dashboard_Submission/datasets/demo/y_test_demo_20k.csv

Also saved to kernel at: /content/demo

Done! 20,000 samples saved (10K benign + 10K attack).


: 